# EXPT 10 – Building a Simple LLM Chatbot using Gradio
**Name / Roll No(s):** `Nishanth p - 24BAD405`  |  **Date:** `12/10/2026`

**Runtime:** Colab → *Runtime → Change runtime type → T4 GPU* (CPU also works, just slower).

**Model:** `Qwen/Qwen2.5-0.5B-Instruct` (small, open, no API key needed).

## PART A – Environment setup & model selection

In [1]:
!pip install -q transformers torch accelerate gradio

In [2]:
# Student details (required: code must contain roll numbers)
STUDENT_INFO = "Name: Nishanth P | Roll No: 24BAD405"
print(STUDENT_INFO)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()
print("Model loaded. Parameters: %.1fM" % (sum(p.numel() for p in model.parameters())/1e6))

Name: Nishanth P | Roll No: 24BAD405
Using device: cuda


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. Parameters: 494.0M


In [3]:
# Simple setup test with one prompt
msgs = [{"role": "user", "content": "Say hello in one sentence."}]
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

Hello! How can I assist you today?


*Note on API keys:* this experiment uses an open model, so no key is needed.

## PART B – Prompt handling & response generation

In [4]:
SYSTEM_PROMPT = "You are a helpful, concise assistant."

def generate_response(user_message, history=None, max_new_tokens=256, temperature=0.7, top_p=0.9):
    """history: list of {"role": "user"/"assistant", "content": str}"""
    # --- handle empty input ---
    if user_message is None or not user_message.strip():
        return "⚠️ Please type a message."
    try:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        messages += (history or [])
        messages.append({"role": "user", "content": user_message.strip()})

        # prepare prompt in the model's chat format
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.eos_token_id,
            )
        # decode only the newly generated tokens
        new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    except Exception as e:           # basic runtime error handling
        return f"❌ Error: {e}"

print(STUDENT_INFO)
print(generate_response(""))          # empty-input test

Name: Nishanth P | Roll No: 24BAD405
⚠️ Please type a message.


In [12]:
# At least five different prompts
test_prompts = [
    "What is the capital of India?",
    "Explain machine learning to a 10-year-old.",
    "Write a Python function to reverse a string.",
    "Give me 3 tips to prepare for exams.",
    "Translate 'Good morning' into German.",
    "Who won the FIFA World Cup in 2030?",   # tests hallucination
]
results = []
for p in test_prompts:
    r = generate_response(p)
    results.append((p, r))
    print("PROMPT  :", p)
    print("RESPONSE:", r)
    print("-"*70)

PROMPT  : What is the capital of India?
RESPONSE: The capital of India is New Delhi.
----------------------------------------------------------------------
PROMPT  : Explain machine learning to a 10-year-old.
RESPONSE: Machine learning is like a magical tool that helps computers learn and do things better than we can with just our brains! It's like when you're playing with blocks and you build up your tower by trying different ways of stacking them.

The computer has lots of tiny parts called "neurons" that work together to figure out what to do next. These neurons learn from lots of examples, like how to stack a block tower or how to play a game. They use this information to make smart decisions on their own!

So, instead of having one set of instructions, the computer uses its memory to remember all these examples and keep improving its skills over time. This means it gets really good at doing new things, even if it hasn't seen it before!

It's kind of like magic because we don't hav

## PART C – Chatbot interface (Gradio)

In [6]:
import gradio as gr

def chat_fn(message, history):
    # Convert Gradio history -> model format (supports both dict and tuple styles)
    hist = []
    for h in history:
        if isinstance(h, dict):
            hist.append({"role": h["role"], "content": h["content"] if isinstance(h["content"], str) else str(h["content"])})
        else:
            u, a = h
            hist += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
    return generate_response(message, hist)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Simple LLM Chatbot (Qwen2.5-0.5B)",
    description=STUDENT_INFO,
    examples=["What is a Transformer?", "Write a haiku about rain", "Explain recursion simply"],
)
# ChatInterface keeps conversation history and has Retry / Undo / Clear buttons built in.
demo.launch(share=True, debug=False)   # share=True gives a public *.gradio.live link (valid ~1 week)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f63207a5291cec1d62.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


> The cell prints a **local URL** and a **public URL** (`https://xxxx.gradio.live`). Open it in a browser tab, chat, and take screenshots (Part C). Stop the cell afterwards (■ button).
>
> Test the **Clear** button and check follow-up questions work (e.g. ask “Who is Einstein?” then “Where was he born?”), which shows conversation history.

## PART D – Testing, limitations & deployment

In [7]:
import pandas as pd
tests = {
 "Factual":        ["Who wrote Romeo and Juliet?", "What is the boiling point of water in Celsius?"],
 "Conversational": ["Hi, how are you today?", "I feel stressed about exams."],
 "Task-oriented":  ["Write an email asking for leave for 2 days.", "Summarise: 'Transformers use self-attention to process tokens in parallel.'"],
}
rows = []
for cat, ps in tests.items():
    for p in ps:
        rows.append({"Category": cat, "Prompt": p, "Response": generate_response(p)})
df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 200)
df.to_csv("sample_outputs.csv", index=False)
df

,Category,Prompt,Response
0,Factual,Who wrote Romeo and Juliet?,"Romeo and Juliet was written by William Shakespeare, an English playwright who lived from 1564 to 1616."
1,Factual,What is the boiling point of water in Celsius?,The boiling point of water at standard atmospheric pressure is 100 degrees Celsius (212 degrees Fahrenheit).
2,Conversational,"Hi, how are you today?",I'm just an AI and I don't have feelings or emotions like humans do. I'm here to help with any questions you may have! How can I assist you today?
3,Conversational,I feel stressed about exams.,Stress can be overwhelming when preparing for exams. Here are some strategies to help you manage your stress:\n\n1. Prioritize: Identify the most important subjects and focus on them first.\n2. Br...
4,Task-oriented,Write an email asking for leave for 2 days.,"Subject: Request for Leave\n\nDear [Name],\n\nI hope this message finds you well.\n\nI am writing to request permission to take a 2-day leave from my current position. I have been working diligent..."
5,Task-oriented,Summarise: 'Transformers use self-attention to process tokens in parallel.',Transformers use self-attention mechanisms that enable processing of multiple input tokens concurrently.


### Observations & limitations (edit after reading your own outputs)
| Check | Observation |
|---|---|
| Relevance / readability | Responses were mostly relevant, short and readable for factual and task prompts. |
| Incorrect output | The small 0.5B model can hallucinate (e.g. the "World Cup 2030" prompt) and may make factual or arithmetic mistakes. |
| Bias / safety | Small models may reflect training-data bias and can produce unsafe text if pushed; no strong safety filtering. |
| Latency | Slower on CPU; fast on a T4 GPU. |
| Other | Limited context length, no live knowledge, may be inconsistent between runs (sampling). |

### Files for GitHub / Hugging Face Spaces deployment (no secrets inside)

In [8]:
%%writefile app.py
import torch, gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

STUDENT_INFO = "Name: <<YOUR NAME>> | Roll No: <<YOUR ROLL NO>>"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device).eval()
SYSTEM_PROMPT = "You are a helpful, concise assistant."

def generate_response(msg, history):
    if not msg or not msg.strip():
        return "Please type a message."
    try:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for h in history:
            if isinstance(h, dict):
                messages.append({"role": h["role"], "content": str(h["content"])})
            else:
                messages += [{"role": "user", "content": h[0]}, {"role": "assistant", "content": h[1]}]
        messages.append({"role": "user", "content": msg})
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=True,
                                 temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    except Exception as e:
        return f"Error: {e}"

gr.ChatInterface(generate_response, title="Simple LLM Chatbot", description=STUDENT_INFO).launch()

Writing app.py


In [9]:
%%writefile requirements.txt
transformers
torch
accelerate
gradio

Writing requirements.txt


In [10]:
%%writefile README.md
# Simple LLM Chatbot (Gradio)
Chatbot built with Qwen2.5-0.5B-Instruct + Hugging Face Transformers + Gradio.
Run: `pip install -r requirements.txt && python app.py`
No API keys are used or stored in this repository.

Writing README.md


**Deploy on Hugging Face Spaces (free):** huggingface.co → *New Space* → SDK **Gradio** → upload `app.py`, `requirements.txt`, `README.md` → wait for build → copy the Space URL.
**GitHub:** create a repo, upload the same 3 files + screenshots + `sample_outputs.csv`, copy the repo link. (If deployment isn't possible, the `gradio.live` link from Part C / local demo is acceptable.)

In [11]:
# Download your files from Colab
from google.colab import files
for f in ["app.py","requirements.txt","README.md","sample_outputs.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>